# BaseAgent - AlzKB

This notebook contains chronical records of BaseAgent's attempt in recreating AlzKB - v2.

In [6]:
%load_ext autoreload
%autoreload 2
import os
import shutil
import sys

from BaseAgent import BaseAgent, AgentTeam, MaxRoundsExceededError
from BaseAgent.agent_spec import AgentSpec


MCP_CONFIG = "examples/mcp_config.yaml"
SKILLS_DIR = "skills"
TEMPLATE_SRC = os.path.expanduser("~/GitHub/alzkb-updater")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
def _print_token_summary(agents: list):
    """Print per-agent and total token counts from accumulated usage_metrics."""
    print("\n=== Token usage ===")
    totals = {"input": 0, "output": 0, "total": 0}
    for agent in agents:
        metrics = agent.usage_metrics
        input_tokens = sum(m.input_tokens or 0 for m in metrics)
        output_tokens = sum(m.output_tokens or 0 for m in metrics)
        total_tokens = sum(m.total_tokens or 0 for m in metrics)
        cost = sum(m.cost or 0.0 for m in metrics)
        cost_str = f"  ${cost:.4f}" if cost else ""
        print(f"  {agent.spec.name}: {input_tokens} in / {output_tokens} out / {total_tokens} total{cost_str}")
        totals["input"] += input_tokens
        totals["output"] += output_tokens
        totals["total"] += total_tokens
    print(f"  {'─' * 40}")
    print(f"  all agents:  {totals['input']} in / {totals['output']} out / {totals['total']} total")

# Set up agentic AI team

In [10]:
# ----- Ontology Agent -----
ontology_agent = BaseAgent(
    spec=AgentSpec(
        name="ontology_agent",
        role=(
            "A biomedical ontology engineer managing the OWL schema and project configuration. "
            "You own data/ontology/alzkb_v2.rdf and config/project.yaml: update disease_scope "
            "(primary_terms, umls_cuis, doid_ids, mesh_ids) and keep node_types/edge_types in "
            "sync with the RDF. Only modify the RDF on explicit request. Never edit Python source files."
        ),
        llm="azure-claude-haiku-4-5",
    ),
    require_approval="never",
)
ontology_agent.add_mcp(MCP_CONFIG)
ontology_agent.add_skill(os.path.join(SKILLS_DIR, "ontology-protocol", "SKILL.md"))

# ----- Database Agent -----
database_agent = BaseAgent(
    spec=AgentSpec(
        name="database_agent",
        role=(
            "A bioinformatics data engineer managing config/databases.yaml. "
            "You evaluate biomedical data sources and set enabled flags for the target disease. "
            "You do not write parsers or ontology mappings."
        ),
        llm="azure-claude-haiku-4-5",
    ),
    require_approval="never",
)
database_agent.add_mcp(MCP_CONFIG)
database_agent.add_skill(os.path.join(SKILLS_DIR, "database-protocol", "SKILL.md"))

# ----- Engineer Agent -----
engineer_agent = BaseAgent(
    spec=AgentSpec(
        name="engineer_agent",
        role=(
            "A Python software engineer writing parsers under src/parsers/. "
            "Each parser inherits from BaseParser and downloads data from one biomedical source, "
            "returning clean pandas DataFrames. Follow the full registration checklist: "
            "src/parsers/__init__.py, src/main.py PARSERS dict, test/eval_parser.py PARSER_CLASS_MAP. "
            "Run `python src/main.py --source <name>` to verify each parser produces TSVs. "
            "You do not modify OWL files or ontology_mappings.yaml."
        ),
        llm="azure-claude-sonnet-4-6",
    ),
    require_approval="never",
)
engineer_agent.add_mcp(MCP_CONFIG)
engineer_agent.add_skill(os.path.join(SKILLS_DIR, "parser-protocol", "SKILL.md"))

# ----- Mapping Agent -----
mapping_agent = BaseAgent(
    spec=AgentSpec(
        name="mapping_agent",
        role=(
            "A knowledge graph mapping specialist owning config/ontology_mappings.yaml. "
        ),
        llm="azure-claude-haiku-4-5",
    ),
    require_approval="never",
)
mapping_agent.add_mcp(MCP_CONFIG)
mapping_agent.add_skill(os.path.join(SKILLS_DIR, "mapping-protocol", "SKILL.md"))

# ----- Memgraph Agent -----
memgraph_agent = BaseAgent(
    spec=AgentSpec(
        name="memgraph_agent",
        role=(
            "A graph database engineer who runs the full pipeline and validates graph export. "
            "Run `python src/main.py` inside the repo to produce data/output/alzkb_v2_populated.rdf, "
        ),
        llm="azure-claude-haiku-4-5",
    ),
    require_approval="never",
)
memgraph_agent.add_mcp(MCP_CONFIG)
memgraph_agent.add_skill(os.path.join(SKILLS_DIR, "memgraph-protocol", "SKILL.md"))

# ----- Evaluator Agent -----
evaluator_agent = BaseAgent(
    spec=AgentSpec(
        name="evaluator_agent",
        role=(
            "A KG quality evaluator who runs eval_after_parser.py, eval_after_ontology.py, "
            "and eval_after_memgraph.py in sequence. Report tier-1 blocking failures "
            "(zero node/edge counts, OWL conformance < 1.0) and overall KG quality. "
            "Flag any blocking failures that must be resolved before the KG is used."
        ),
        llm="azure-claude-haiku-4-5",
    ),
    require_approval="never",
)
evaluator_agent.add_mcp(MCP_CONFIG)
evaluator_agent.add_skill(os.path.join(SKILLS_DIR, "evaluation-protocol", "SKILL.md"))

Auto-detecting source from model name: azure-claude-haiku-4-5
Creating AnthropicFoundry model: azure-claude-haiku-4-5

🔧 BASE AGENT CONFIGURATION
Agent Name: ontology_agent
LLM: azure-claude-haiku-4-5
Source: AnthropicFoundry
Base URL: None
Loaded Tools: ['run_python_repl', 'read_function_source_code']
Use Tool Retriever: False

Discovered 14 tools from filesystem MCP server (local)
Skill 'ontology-protocol' successfully added
Auto-detecting source from model name: azure-claude-haiku-4-5
Creating AnthropicFoundry model: azure-claude-haiku-4-5

🔧 BASE AGENT CONFIGURATION
Agent Name: database_agent
LLM: azure-claude-haiku-4-5
Source: AnthropicFoundry
Base URL: None
Loaded Tools: ['run_python_repl', 'read_function_source_code']
Use Tool Retriever: False

Discovered 14 tools from filesystem MCP server (local)
Skill 'database-protocol' successfully added
Auto-detecting source from model name: azure-claude-sonnet-4-6
Creating AnthropicFoundry model: azure-claude-sonnet-4-6

🔧 BASE AGENT CONF

Skill(name='evaluation-protocol', description='Use when running, interpreting, or extending the KG build evaluation suite. Covers the three pipeline-stage eval scripts (eval_after_parser.py, eval_after_ontology.py, eval_after_memgraph.py), output JSON format, the three-tier metric system, blocking vs. monitoring thresholds, and adding new metrics. Use when asked to evaluate pipeline output, diagnose zero-count or low-resolution failures, check ontology conformance, run benchmarks, or interpret eval reports.', tools=[], instructions='## Eval Scripts by Pipeline Stage\n\nRun each script after its prerequisite pipeline step completes.\n\n| Script | Prerequisite | Input files |\n|--------|-------------|-------------|\n| `eval/eval_after_parser.py` | `python src/main.py` (all enabled sources) | `data/processed/<source>/<name>.tsv` |\n| `eval/eval_after_ontology.py` | populate step | `data/output/alzkb_v2_populated.rdf`, `data/ontology/alzkb_v2.rdf` |\n| `eval/eval_after_memgraph.py` | expor

# Evaluate parsers

In [11]:

agents = [
    database_agent, engineer_agent, evaluator_agent,
]

team = AgentTeam(
    agents=agents,
    supervisor_llm="azure-claude-opus-4-6",
    max_rounds=20,
)

task = (
    f"Evaluate the AOP-DB parser for the Alzheimer's disease in the repo at {TEMPLATE_SRC}.\n\n"
    "Coordinate tasks across the following agents:\n"
    "1. database_agent: Update config/databases.yaml — enable sources relevant to the "
    "disease and disable irrelevant ones. Report the list of newly enabled sources.\n"
    "2. engineer_agent: Add new or update parsers in src/parsers/, "
    "write a BaseParser subclass and complete the full registration checklist. "
    "Verify with `python src/main.py --source <name>` run inside the repo.\n"
    "3. evaluator_agent: Run all three eval scripts and report the quality summary. "
    "List any tier-1 blocking failures explicitly.\n\n"
    "Pipeline contracts — violations fail silently:\n"
    "- The databases.yaml key, PARSERS key, PARSER_CLASS_MAP key, ontology_mappings.yaml "
    "prefix, and data/processed/ subdirectory name must all be identical strings.\n"
    "- In ontology_mappings.yaml, all node entries must precede all relationship entries.\n"
    "- OWL names in ontology_mappings.yaml must be active (uncommented) in project.yaml.\n"
    "- Credentials use the _env suffix in databases.yaml args; the parser constructor "
    "must accept the stripped parameter name.\n"
)

Auto-detecting source from model name: azure-claude-opus-4-6
Creating AnthropicFoundry model: azure-claude-opus-4-6


In [12]:
_log, result = team.run_sync(task)
print(f"\n=== Result ===\n{result}")


[evaluator_agent] Starting
================================ Human Message =================================

The task is to evaluate the existing AOP-DB parser for Alzheimer's disease in the repo at /Users/lib/GitHub/alzkb-updater. No new parsers need to be written and no config changes are requested — we just need an evaluation.

Please do the following:

1. Change directory to /Users/lib/GitHub/alzkb-updater.
2. Run all three evaluation scripts in order:
   a. python test/eval_after_parser.py — checks parser output (TSVs in data/processed/).
   b. python test/eval_after_ontology.py — checks OWL/ontology conformance of the parsed data.
   c. python test/eval_after_memgraph.py — checks the final KG loaded into Memgraph.
3. For each script, capture and report the full output.
4. Provide a quality summary covering:
   - Node and edge counts (zero counts are tier-1 blocking failures).
   - OWL conformance score (anything < 1.0 is a tier-1 blocking failure).
   - Any other warnings or err